In [ ]:
# ============================================
# 02_Preprocessing.ipynb
# PREPROCESSING DATASET
# ============================================

import os
import shutil
import random
from pathlib import Path
from PIL import Image
import numpy as np
import cv2
from tqdm import tqdm
import json
import sys

# Set seed untuk reproducibility
random.seed(42)
np.random.seed(42)

print("="*60)
print("🔄 PREPROCESSING DATASET")
print("="*60)

# ============================================
# AUTO DETECT PROJECT ROOT & DATASET
# ============================================

def find_project_root():
    """Mencari root project secara otomatis"""
    current = Path.cwd()
    
    # Cek apakah kita di folder notebooks atau root
    if current.name == "notebooks" or current.name == "notebook":
        return current.parent
    
    # Cek apakah ada folder .venv di parent
    if (current / ".venv").exists():
        return current
    
    # Cari ke atas sampai menemukan .venv atau requirements.txt
    for parent in current.parents:
        if (parent / ".venv").exists() or (parent / "requirements.txt").exists():
            return parent
    
    return current

def find_dataset():
    """Mencari folder dataset secara otomatis"""
    
    project_root = find_project_root()
    
    possible_paths = [
        project_root / "dataset" / "raw",           # dataset/raw/
        project_root / "dataset",                    # dataset/
        project_root / "notebook" / "dataset" / "raw", # notebook/dataset/raw/
        project_root / "notebook" / "dataset",       # notebook/dataset/
        project_root / "data" / "raw",               # data/raw/
        project_root / "data",                       # data/
        Path.cwd() / "dataset" / "raw",              # current/dataset/raw/
        Path.cwd() / "dataset",                      # current/dataset/
    ]
    
    for path in possible_paths:
        if path.exists():
            # Cek folder Drowsy dan Non Drowsy
            drowsy = path / "Drowsy"
            non_drowsy = path / "Non Drowsy"
            if drowsy.exists() and non_drowsy.exists():
                return path, drowsy, non_drowsy
    
    # Jika tidak ditemukan, cari manual
    print("🔍 Mencari dataset di seluruh folder...")
    for root, dirs, files in os.walk(project_root):
        if "Drowsy" in dirs and "Non Drowsy" in dirs:
            dataset_path = Path(root)
            drowsy = dataset_path / "Drowsy"
            non_drowsy = dataset_path / "Non Drowsy"
            return dataset_path, drowsy, non_drowsy
    
    print("❌ DATASET TIDAK DITEMUKAN!")
    print("\n📁 Struktur yang dicari:")
    print("   dataset/")
    print("   └── raw/")
    print("       ├── Drowsy/")
    print("       └── Non Drowsy/")
    print("\n💡 Pastikan folder dataset ada di salah satu lokasi:")
    for p in possible_paths:
        print(f"   - {p}")
    raise FileNotFoundError("Dataset tidak ditemukan!")

# ============================================
# SETUP PATHS
# ============================================

project_root = find_project_root()
dataset_path, drowsy_folder, non_drowsy_folder = find_dataset()
processed_dir = project_root / "dataset" / "processed"

print(f"📁 Project root: {project_root}")
print(f"📁 Dataset path: {dataset_path}")
print(f"📁 Processed path: {processed_dir}")

# ============================================
# BUAT FOLDER PROCESSED
# ============================================
print("\n[1] Membuat struktur folder...")

# Hapus folder lama jika ada
if processed_dir.exists():
    shutil.rmtree(processed_dir)
    print("   - Folder processed lama dihapus")

# Buat folder baru
for model_type in ['cnn_mobilenet', 'yolo']:
    for split in ['train', 'valid', 'test']:
        for class_name in ['awake', 'drowsy']:
            path = processed_dir / model_type / split / class_name
            path.mkdir(parents=True, exist_ok=True)

print("✅ Struktur folder berhasil dibuat")

# ============================================
# HITUNG JUMLAH GAMBAR
# ============================================
print("\n[2] Menghitung jumlah gambar...")

def count_images(folder):
    """Menghitung jumlah gambar dalam folder"""
    if not folder:
        return 0
    return len(list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg')))

drowsy_count = count_images(drowsy_folder)
non_drowsy_count = count_images(non_drowsy_folder)
total_images = drowsy_count + non_drowsy_count

print(f"   - Drowsy: {drowsy_count} gambar")
print(f"   - Non Drowsy: {non_drowsy_count} gambar")
print(f"   - Total: {total_images} gambar")

# ============================================
# FUNGSI PREPROCESSING
# ============================================
print("\n[3] Fungsi preprocessing...")

def preprocess_image(image_path, target_size=(224, 224)):
    """
    Preprocess gambar:
    1. Resize dengan maintain aspect ratio
    2. Pad ke target size (center crop)
    3. Convert ke RGB
    """
    try:
        # Buka gambar
        img = Image.open(image_path)
        
        # Convert ke RGB jika perlu
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Resize dengan maintain aspect ratio
        img.thumbnail(target_size, Image.Resampling.LANCZOS)
        
        # Buat canvas kosong dengan target size
        new_img = Image.new('RGB', target_size, (0, 0, 0))
        
        # Paste gambar di tengah canvas
        x = (target_size[0] - img.size[0]) // 2
        y = (target_size[1] - img.size[1]) // 2
        new_img.paste(img, (x, y))
        
        return new_img
    except Exception as e:
        print(f"⚠️ Error processing {image_path.name}: {e}")
        return None

# ============================================
# SPLIT DAN PROSES DATA
# ============================================
print("\n[4] Split dataset (train:valid:test = 70:15:15)...")

split_ratio = (0.7, 0.15, 0.15)

# Mapping folder ke nama kelas output
class_mapping = {
    drowsy_folder: 'drowsy',
    non_drowsy_folder: 'awake'
}

# Hitung total gambar per split
total_train = 0
total_valid = 0
total_test = 0

for folder, class_name in class_mapping.items():
    if not folder or not folder.exists():
        print(f"⚠️ Folder tidak ditemukan: {folder}")
        continue
    
    # Dapatkan semua gambar
    image_files = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
    random.shuffle(image_files)
    
    total = len(image_files)
    if total == 0:
        print(f"⚠️ Tidak ada gambar di {folder.name}")
        continue
    
    train_end = int(total * split_ratio[0])
    valid_end = train_end + int(total * split_ratio[1])
    
    splits = {
        'train': image_files[:train_end],
        'valid': image_files[train_end:valid_end],
        'test': image_files[valid_end:]
    }
    
    print(f"\n📁 {folder.name} -> {class_name}:")
    print(f"   Total: {total} gambar")
    
    for split_name, files in splits.items():
        # ===== UNTUK CNN & MOBILENETV2 (dengan preprocessing) =====
        dest_dir = processed_dir / 'cnn_mobilenet' / split_name / class_name
        
        # Gunakan tqdm untuk progress bar
        for img_path in tqdm(files, desc=f"  {split_name} (CNN)", leave=False):
            processed_img = preprocess_image(img_path)
            if processed_img is not None:
                processed_img.save(dest_dir / img_path.name)
        
        # ===== UNTUK YOLO (copy langsung tanpa preprocessing) =====
        dest_dir_yolo = processed_dir / 'yolo' / split_name / class_name
        for img_path in tqdm(files, desc=f"  {split_name} (YOLO)", leave=False):
            shutil.copy2(img_path, dest_dir_yolo / img_path.name)
        
        print(f"  ✅ {split_name}: {len(files)} gambar")
        
        # Akumulasi total
        if split_name == 'train':
            total_train += len(files)
        elif split_name == 'valid':
            total_valid += len(files)
        elif split_name == 'test':
            total_test += len(files)

# ============================================
# VERIFIKASI HASIL
# ============================================
print("\n[5] Verifikasi hasil preprocessing...")

print("\n📊 CNN_MOBILENET:")
for split in ['train', 'valid', 'test']:
    awake_count = len(list((processed_dir / 'cnn_mobilenet' / split / 'awake').glob('*')))
    drowsy_count = len(list((processed_dir / 'cnn_mobilenet' / split / 'drowsy').glob('*')))
    print(f"   {split}: awake={awake_count}, drowsy={drowsy_count}")

print("\n📊 YOLO:")
for split in ['train', 'valid', 'test']:
    awake_count = len(list((processed_dir / 'yolo' / split / 'awake').glob('*')))
    drowsy_count = len(list((processed_dir / 'yolo' / split / 'drowsy').glob('*')))
    print(f"   {split}: awake={awake_count}, drowsy={drowsy_count}")

# ============================================
# BUAT DATASET.YAML UNTUK YOLO
# ============================================
print("\n[6] Membuat dataset.yaml untuk YOLO...")

yolo_data = {
    'path': str(processed_dir.absolute()),
    'train': 'yolo/train',
    'val': 'yolo/valid',
    'test': 'yolo/test',
    'nc': 2,
    'names': ['awake', 'drowsy']
}

yaml_path = processed_dir / 'dataset.yaml'
import yaml
with open(yaml_path, 'w') as f:
    yaml.dump(yolo_data, f, default_flow_style=False)

print(f"✅ dataset.yaml dibuat di: {yaml_path}")

# ============================================
# SIMPAN INFORMASI PREPROCESSING
# ============================================
print("\n[7] Menyimpan informasi preprocessing...")

preprocess_info = {
    "project_root": str(project_root),
    "dataset_path": str(dataset_path),
    "processed_path": str(processed_dir),
    "drowsy_count": drowsy_count,
    "non_drowsy_count": non_drowsy_count,
    "total_images": total_images,
    "split_ratio": split_ratio,
    "target_size": "224x224",
    "split_counts": {
        "train": total_train,
        "valid": total_valid,
        "test": total_test
    },
    "class_mapping": {
        "Drowsy": "drowsy",
        "Non Drowsy": "awake"
    }
}

with open(processed_dir / "preprocess_info.json", 'w') as f:
    json.dump(preprocess_info, f, indent=4)

print(f"✅ Informasi preprocessing disimpan di: {processed_dir / 'preprocess_info.json'}")

# ============================================
# RINGKASAN AKHIR
# ============================================
print("\n" + "="*60)
print("✅ PREPROCESSING SELESAI!")
print("="*60)

print("\n📊 RINGKASAN:")
print(f"   - Total gambar: {total_images}")
print(f"   - Drowsy: {drowsy_count} gambar")
print(f"   - Non Drowsy: {non_drowsy_count} gambar")
print(f"   - Train: {total_train} gambar")
print(f"   - Valid: {total_valid} gambar")
print(f"   - Test: {total_test} gambar")

print(f"\n📁 Dataset siap digunakan di: {processed_dir}")

# Cek struktur akhir
print("\n📂 Struktur folder processed:")
for model_type in ['cnn_mobilenet', 'yolo']:
    print(f"   {model_type}/")
    for split in ['train', 'valid', 'test']:
        awake_count = len(list((processed_dir / model_type / split / 'awake').glob('*')))
        drowsy_count = len(list((processed_dir / model_type / split / 'drowsy').glob('*')))
        print(f"      {split}/")
        print(f"         awake/ ({awake_count} images)")
        print(f"         drowsy/ ({drowsy_count} images)")

print("\n" + "="*60)
print("✅ PREPROCESSING SELESAI! LANJUT KE TRAINING MODEL!")
print("="*60)